# 1 - Mount Drive and Verify Project Structure

Run this cell at the start of every new Colab session.

It is safe to rerun — all `mkdir` calls are idempotent.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=False)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'


In [ ]:
DIRS = [
    'checkpoints',
    'datasets/stage_a',       # tiny handmade data
    'datasets/stage_b',       # small real data (Tatoeba)
    'datasets/stage_c',       # medium subset
    'datasets/stage_d',       # WMT14
    'vocab',
    'logs',
    'results',
    'translations',
    'figures/attention',
    'configs',
    'experiments',
]

In [ ]:
for dir in DIRS:
    path = os.path.join(PROJECT_ROOT, dir)
    os.makedirs(path, exist_ok=True)

print("Google Drive mounted.")
print(f"Project root: {PROJECT_ROOT}")
print("\nDirectory structure:")
for dirpath, dirnames, filenames in os.walk(PROJECT_ROOT):
    depth = dirpath.replace(PROJECT_ROOT, '').count(os.sep)
    indent = '  ' * depth
    print(f"{indent}{os.path.basename(dirpath)}/")
    if depth < 2:
        for f in filenames:
            print(f"{indent}  {f}")

Google Drive mounted.
Project root: /content/drive/MyDrive/Transformer_Mine

Directory structure:
Transformer_Mine/
  02_data_preprocessing.ipynb
  directory.txt.gdoc
  01_environment_setup.ipynb
  03_tokenization.ipynb
  checkpoints/
  datasets/
    stage_a/
    stage_b/
    stage_c/
    stage_d/
  vocab/
  logs/
  results/
  translations/
  figures/
    attention/
  configs/
  experiments/


# 2 - Tiny Handmade Translation Data

## Purpose

A micro-dataset of approximately **20 sentence pairs** used **only** to verify that the full pipeline runs without errors:

**Tokenize → Encode → Pad → Batch → Mask → Model → Loss**

> **Important:** This is **not a training dataset**.  
> The model is expected to overfit this dataset in **Phase 13**.

In [ ]:
import os

In [ ]:
STAGE_A_DIR = os.path.join(PROJECT_ROOT, 'datasets/stage_a')

In [ ]:
stage_a_pairs = [
    ("I am a student .", "Ich bin ein Student ."),
    ("She is a teacher .", "Sie ist eine Lehrerin ."),
    ("He is a doctor .", "Er ist ein Arzt ."),
    ("We are friends .", "Wir sind Freunde ."),
    ("They are students .", "Sie sind Studenten ."),
    ("I like music .", "Ich mag Musik ."),
    ("She likes books .", "Sie mag Bücher ."),
    ("He likes sports .", "Er mag Sport ."),
    ("I go to school .", "Ich gehe zur Schule ."),
    ("She goes to work .", "Sie geht zur Arbeit ."),
    ("He goes home .", "Er geht nach Hause ."),
    ("I have a dog .", "Ich habe einen Hund ."),
    ("She has a cat .", "Sie hat eine Katze ."),
    ("We have a car .", "Wir haben ein Auto ."),
    ("I am happy .", "Ich bin glücklich ."),
    ("She is sad .", "Sie ist traurig ."),
    ("He is tired .", "Er ist müde ."),
    ("The cat is small .", "Die Katze ist klein ."),
    ("The dog is big .", "Der Hund ist groß ."),
    ("The book is new .", "Das Buch ist neu ."),
]

In [ ]:
len(stage_a_pairs)

20

In [ ]:
src_path = os.path.join(STAGE_A_DIR, 'train.en')
tgt_path = os.path.join(STAGE_A_DIR, 'train.de')

In [ ]:
with open(src_path, 'w', encoding='utf-8') as src_file,\
     open(tgt_path, 'w', encoding='utf-8') as tgt_file:
    for src, tgt in stage_a_pairs:
        src_file.write(f'{src}\n')
        tgt_file.write(f'{tgt}\n')


In [ ]:
print(f"   {src_path}")

   /content/drive/MyDrive/Transformer_Mine/datasets/stage_a/train.en


In [ ]:
print(f"   {tgt_path}")

   /content/drive/MyDrive/Transformer_Mine/datasets/stage_a/train.de


In [ ]:
print("Sample pairs:")
print(f"{'English':<35} {'German'}")
print("-" * 70)
for en, de in stage_a_pairs[:5]:
    print(f"{en:<35} {de}")

Sample pairs:
English                             German
----------------------------------------------------------------------
I am a student .                    Ich bin ein Student .
She is a teacher .                  Sie ist eine Lehrerin .
He is a doctor .                    Er ist ein Arzt .
We are friends .                    Wir sind Freunde .
They are students .                 Sie sind Studenten .


# 3 - Tatoeba EN→DE via OPUS

Downloads a small real parallel corpus of approximately **200k sentence pairs** that fits comfortably in Colab RAM and downloads in **under 2 minutes**.

## Source

**OPUS — Open Parallel Corpus, Tatoeba subset**

Source URL:  
https://object.oslo.systems/opus/Tatoeba/v2023-04-12/moses/de-en.txt.zip

## Why Tatoeba?

Tatoeba is a relatively clean, sentence-aligned corpus suitable for quick experiments before attempting larger datasets such as **WMT14**.

## Expected Dataset Size

Approximately **90,000 EN-DE sentence pairs** are expected in the Tatoeba corpus.

> The exact count may vary depending on the corpus release.

In [ ]:
import zipfile
import urllib.request

In [ ]:
STAGE_B_DIR = os.path.join(PROJECT_ROOT, 'datasets/stage_b')

In [ ]:
ZIP_PATH = os.path.join(STAGE_B_DIR, 'tatoeba_de-en.zip')
OPUS_URL = (
    "https://object.pouta.csc.fi/OPUS-Tatoeba/v2023-04-12/moses/de-en.txt.zip"
)

In [ ]:
def download_with_progress(url, dest_path):
    """Download a file, showing MB downloaded."""

    def _progress(count, block_size, total_size):
        downloaded = count * block_size / 1_048_576
        total      = total_size / 1_048_576
        print(f"\r  Downloading: {downloaded:.1f} / {total:.1f} MB", end='')

    urllib.request.urlretrieve(url, dest_path, reporthook=_progress)
    print()


In [ ]:
if not os.path.exists(ZIP_PATH):
    print(f"Downloading Tatoeba EN-DE from OPUS …")
    download_with_progress(OPUS_URL, ZIP_PATH)
    print(f"Downloaded to {ZIP_PATH}")
else:
    print(f"Zip already exists: {ZIP_PATH}")

  Downloading: 10.0 / 10.0 MB
Downloaded to /content/drive/MyDrive/Transformer_Mine/datasets/stage_b/tatoeba_de-en.zip


In [ ]:
en_raw = os.path.join(STAGE_B_DIR, 'Tatoeba.de-en.en')
de_raw = os.path.join(STAGE_B_DIR, 'Tatoeba.de-en.de')

In [ ]:
if not (os.path.exists(en_raw) and os.path.exists(de_raw)):
    print("Extracting …")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        print("Files in zip:", z.namelist())
        z.extractall(STAGE_B_DIR)
    print(f"Extracted to {STAGE_B_DIR}")
else:
    print(f"Already extracted.")


Extracting …
Files in zip: ['README', 'LICENSE', 'Tatoeba.de-en.de', 'Tatoeba.de-en.en', 'Tatoeba.de-en.xml']
Extracted to /content/drive/MyDrive/Transformer_Mine/datasets/stage_b


In [ ]:
with open(en_raw, encoding='utf-8') as f:
    en_lines = f.readlines()

In [ ]:
with open(de_raw, encoding='utf-8') as f:
    de_lines = f.readlines()

In [ ]:
assert len(en_lines) == len(de_lines), \
    f"Line count mismatch: EN={len(en_lines)}, DE={len(de_lines)}"

In [ ]:
print(f"\nTotal sentence pairs: {len(en_lines):,}")
print("\nSample pairs:")
print(f"{'English':<50} {'German'}")
for i in range(5):
    print(f"{en_lines[i].strip():<50} {de_lines[i].strip()}")


Total sentence pairs: 322,413

Sample pairs:
English                                            German
Let's try something.                               Lass uns etwas versuchen!
I have to go to sleep.                             Ich muss schlafen gehen.
What is it?                                        Was ist das?
What's that?                                       Was ist das?
What is this?                                      Was ist das?


# 4 - Filter, Clean, and Split

The following heuristic filtering rules are applied:

1. Remove pairs where either side is empty.

2. Remove pairs where either side exceeds `MAX_LEN` tokens.  
   Whitespace splitting is used as a proxy for token count at this stage, since subword tokenization has not yet been introduced.

3. Remove pairs where the length ratio exceeds `MAX_RATIO`.  
   For example, if one side is **10× longer** than the other, the pair is likely to be misaligned or noisy.

4. Shuffle the dataset using a fixed seed for reproducibility.

5. Split the cleaned dataset into **train / validation / test** sets.

The resulting files are stored in:

```text
datasets/stage_b/
├── train.en
├── train.de
├── val.en
├── val.de
├── test.en
└── test.de

In [ ]:
import random

In [ ]:
random.seed(42)

In [ ]:
MAX_LEN   = 100
MAX_RATIO = 4.0
VAL_SIZE  = 2_000
TEST_SIZE = 2_000

In [ ]:
STAGE_B_DIR = os.path.join(PROJECT_ROOT, 'datasets/stage_b')
en_raw = os.path.join(STAGE_B_DIR, 'Tatoeba.de-en.en')
de_raw = os.path.join(STAGE_B_DIR, 'Tatoeba.de-en.de')

In [ ]:
STAGE_B_DIR, en_raw, de_raw

('/content/drive/MyDrive/Transformer_Mine/datasets/stage_b',
 '/content/drive/MyDrive/Transformer_Mine/datasets/stage_b/Tatoeba.de-en.en',
 '/content/drive/MyDrive/Transformer_Mine/datasets/stage_b/Tatoeba.de-en.de')

In [ ]:
with open(en_raw, encoding='utf-8') as f:
    en_lines = [l.strip() for l in f]

In [ ]:
with open(de_raw, encoding='utf-8') as f:
    de_lines = [l.strip() for l in f]

In [ ]:
filtered_pairs = []
stats = {'empty': 0, 'too_long': 0, 'ratio': 0, 'kept': 0}

In [ ]:
for en, de in zip(en_lines, de_lines):
    # 1. Missing / Empty Sentences
    if not en or not de:
        stats['empty'] += 1
        continue
        #If either the English or German sentence is empty, it skips the pair and increments stats['empty']

    # 2. Sentence Length Limits
    en_len = len(en.split())
    de_len = len(de.split())
    if en_len > MAX_LEN or de_len > MAX_LEN:
        stats['too_long'] += 1
        continue

    # 3. Length Imbalance Ratio
    ratio = max(en_len, de_len) / min(en_len, de_len, 1)
    if ratio > MAX_RATIO:
        stats['ratio'] += 1
        continue

    filtered_pairs.append((en, de))
    stats['kept'] += 1

In [ ]:
print(f"Filtering report:")
print(f"  Total raw pairs  : {len(en_lines):>8,}")
print(f"  Removed (empty)  : {stats['empty']:>8,}")
print(f"  Removed (too long): {stats['too_long']:>8,}")
print(f"  Removed (ratio)  : {stats['ratio']:>8,}")
print(f"  Kept             : {stats['kept']:>8,}")
print()

Filtering report:
  Total raw pairs  :  322,413
  Removed (empty)  :        7
  Removed (too long):        7
  Removed (ratio)  :  273,818
  Kept             :   48,581



In [ ]:
random.shuffle(filtered_pairs)

In [ ]:
test_pairs  = filtered_pairs[:TEST_SIZE]
val_pairs   = filtered_pairs[TEST_SIZE : TEST_SIZE + VAL_SIZE]
train_pairs = filtered_pairs[TEST_SIZE + VAL_SIZE :]

In [ ]:
len(test_pairs), len(val_pairs), len(train_pairs)

(2000, 2000, 44581)

In [ ]:
def write_split(pairs, prefix):
    """
    Write a list of (en, de) pairs to prefix.en and prefix.de.
    """
    with open(prefix + '.en', 'w', encoding='utf-8') as f_en, \
         open(prefix + '.de', 'w', encoding='utf-8') as f_de:
        for en, de in pairs:
            f_en.write(en + '\n')
            f_de.write(de + '\n')

    print(f"  Written {len(pairs):>6,} pairs → {prefix}.{{en,de}}")

In [ ]:
print("Writing splits:")
write_split(train_pairs, os.path.join(STAGE_B_DIR, 'train'))
write_split(val_pairs,   os.path.join(STAGE_B_DIR, 'val'))
write_split(test_pairs,  os.path.join(STAGE_B_DIR, 'test'))

print(f"   Train: {len(train_pairs):,}")
print(f"   Val  : {len(val_pairs):,}")
print(f"   Test : {len(test_pairs):,}")

Writing splits:
  Written 44,581 pairs → /content/drive/MyDrive/Transformer_Mine/datasets/stage_b/train.{en,de}
  Written  2,000 pairs → /content/drive/MyDrive/Transformer_Mine/datasets/stage_b/val.{en,de}
  Written  2,000 pairs → /content/drive/MyDrive/Transformer_Mine/datasets/stage_b/test.{en,de}
   Train: 44,581
   Val  : 2,000
   Test : 2,000


# 5 - Dataset Registry and Active Config

A single place to switch between dataset stages without hunting through the notebook.

Set `ACTIVE_STAGE` to `'A'`, `'B'`, `'C'`, or `'D'` throughout the project.

All downstream cells read from `DATASET_CONFIG`.

In [ ]:
import json

In [ ]:
DATASET_REGISTRY = {
    'A': {
        'name'        : 'Stage A — Tiny Handmade',
        'train_src'   : os.path.join(PROJECT_ROOT, 'datasets/stage_a/train.en'),
        'train_tgt'   : os.path.join(PROJECT_ROOT, 'datasets/stage_a/train.de'),
        'val_src'     : os.path.join(PROJECT_ROOT, 'datasets/stage_a/train.en'),
        'val_tgt'     : os.path.join(PROJECT_ROOT, 'datasets/stage_a/train.de'),
        'test_src'    : os.path.join(PROJECT_ROOT, 'datasets/stage_a/train.en'),
        'test_tgt'    : os.path.join(PROJECT_ROOT, 'datasets/stage_a/train.de'),
        'vocab_size'  : None,
        'max_len'     : 20,
        'batch_size'  : 4,
        'purpose'     : 'Pipeline smoke test. Model must overfit this.'
    },
    'B': {
        'name'        : 'Stage B — Tatoeba EN-DE',
        'train_src'   : os.path.join(PROJECT_ROOT, 'datasets/stage_b/train.en'),
        'train_tgt'   : os.path.join(PROJECT_ROOT, 'datasets/stage_b/train.de'),
        'val_src'     : os.path.join(PROJECT_ROOT, 'datasets/stage_b/val.en'),
        'val_tgt'     : os.path.join(PROJECT_ROOT, 'datasets/stage_b/val.de'),
        'test_src'    : os.path.join(PROJECT_ROOT, 'datasets/stage_b/test.en'),
        'test_tgt'    : os.path.join(PROJECT_ROOT, 'datasets/stage_b/test.de'),
        'vocab_size'  : 8000,
        'max_len'     : 100,
        'batch_size'  : 64,
        'purpose'     : 'Small real data. Validate BPE vocab, batching, BLEU.'
    },
    'C': {
        'name'        : 'Stage C — WMT14 Subset (≤500k pairs)',
        'train_src'   : os.path.join(PROJECT_ROOT, 'datasets/stage_c/train.en'),
        'train_tgt'   : os.path.join(PROJECT_ROOT, 'datasets/stage_c/train.de'),
        'val_src'     : os.path.join(PROJECT_ROOT, 'datasets/stage_c/val.en'),
        'val_tgt'     : os.path.join(PROJECT_ROOT, 'datasets/stage_c/val.de'),
        'test_src'    : os.path.join(PROJECT_ROOT, 'datasets/stage_c/test.en'),
        'test_tgt'    : os.path.join(PROJECT_ROOT, 'datasets/stage_c/test.de'),
        'vocab_size'  : 32000,
        'max_len'     : 128,
        'batch_size'  : 128,
        'purpose'     : 'Colab-scale training run with real WMT data subset.'
    },
    'D': {
        'name'        : 'Stage D — WMT14 Full EN→DE',
        'train_src'   : os.path.join(PROJECT_ROOT, 'datasets/stage_d/train.en'),
        'train_tgt'   : os.path.join(PROJECT_ROOT, 'datasets/stage_d/train.de'),
        'val_src'     : os.path.join(PROJECT_ROOT, 'datasets/stage_d/newstest2013.en'),
        'val_tgt'     : os.path.join(PROJECT_ROOT, 'datasets/stage_d/newstest2013.de'),
        'test_src'    : os.path.join(PROJECT_ROOT, 'datasets/stage_d/newstest2014.en'),
        'test_tgt'    : os.path.join(PROJECT_ROOT, 'datasets/stage_d/newstest2014.de'),
        'vocab_size'  : 37000,     # paper: ~37k shared BPE vocabulary
        'max_len'     : 128,
        'batch_size'  : 128,
        'purpose'     : 'Paper target. Full WMT14 EN-DE. ~4.5M pairs.'
    },
}

In [ ]:
ACTIVE_STAGE = 'A'

In [ ]:
DATASET_CONFIG = DATASET_REGISTRY[ACTIVE_STAGE]

In [ ]:
DATASET_CONFIG

{'name': 'Stage A — Tiny Handmade',
 'train_src': '/content/drive/MyDrive/Transformer_Mine/datasets/stage_a/train.en',
 'train_tgt': '/content/drive/MyDrive/Transformer_Mine/datasets/stage_a/train.de',
 'val_src': '/content/drive/MyDrive/Transformer_Mine/datasets/stage_a/train.en',
 'val_tgt': '/content/drive/MyDrive/Transformer_Mine/datasets/stage_a/train.de',
 'test_src': '/content/drive/MyDrive/Transformer_Mine/datasets/stage_a/train.en',
 'test_tgt': '/content/drive/MyDrive/Transformer_Mine/datasets/stage_a/train.de',
 'vocab_size': None,
 'max_len': 20,
 'batch_size': 4,
 'purpose': 'Pipeline smoke test. Model must overfit this.'}

In [ ]:
print(f"  ACTIVE DATASET STAGE: {ACTIVE_STAGE}")
print(f"  {DATASET_CONFIG['name']}")
print(f"{'='*60}")
print(f"  Purpose     : {DATASET_CONFIG['purpose']}")
print(f"  Vocab size  : {DATASET_CONFIG['vocab_size']}")
print(f"  Max length  : {DATASET_CONFIG['max_len']}")
print(f"  Batch size  : {DATASET_CONFIG['batch_size']}")

  ACTIVE DATASET STAGE: A
  Stage A — Tiny Handmade
  Purpose     : Pipeline smoke test. Model must overfit this.
  Vocab size  : None
  Max length  : 20
  Batch size  : 4


In [ ]:
config_path = os.path.join(PROJECT_ROOT, 'configs/dataset_config.json')
with open(config_path, 'w') as f:
    json.dump({'active_stage': ACTIVE_STAGE, **DATASET_CONFIG}, f, indent=2)

# 6 - Data Loading Utilities

Functions that:

1. Load raw text pairs from disk.
2. Apply a user-supplied `encode()` function to each line.
3. Pad sequences to a fixed length.
4. Build a `tf.data.Dataset` with padding and batching.

These functions are **tokenizer-agnostic** at this stage. They accept a callable:

`encode(text) → list[int]`

**Phase 3** will supply the real **BPE tokenizer**.

For now, we test these utilities with a **dummy whitespace tokenizer** so that the complete pipeline can be verified end-to-end before the vocabulary is built.

In [ ]:
import numpy as np
import tensorflow as tf
from typing import Callable, List, Tuple

In [ ]:
PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3

In [ ]:
def load_raw_pairs(src_path: str, tgt_path: str) -> Tuple[List[str], List[str]]:
    """
    Load source and target lines from two parallel text files.

    Returns:
        src_lines: list of raw source strings
        tgt_lines: list of raw target strings

    Raises AssertionError if line counts differ.
    """

    with open(src_path, encoding='utf-8') as f:
        src_lines = [l.strip() for l in f if l.strip()]
    with open(tgt_path, encoding='utf-8') as f:
        tgt_lines = [l.strip() for l in f if l.strip()]

    assert len(src_lines) == len(tgt_lines), (
        f"Source ({len(src_lines)}) and target ({len(tgt_lines)}) "
        f"line counts do not match."
    )

    return src_lines, tgt_lines

In [ ]:
def encode_and_wrap(
    text: str,
    encode_fn: Callable[[str], List[int]],
    bos_id: int = BOS_ID,
    eos_id: int = EOS_ID,
) -> List[int]:
    """
    Encode a sentence and prepend <BOS>, append <EOS>.

    Args:
        text    : raw string, e.g. "I am a student ."
        encode_fn: callable that maps a string → list[int]
        bos_id  : token ID for beginning-of-sequence
        eos_id  : token ID for end-of-sequence

    Returns:
        [BOS_ID, id1, id2, ..., EOS_ID]
    """

    ids = encode_fn(text)
    return [bos_id] + ids + [eos_id]

BOS_ID = 1   BOS

EOS_ID = 2   EOS

Tokenization step (encode_fn("I am happy")):
Returns [10, 25, 88]

Wrapping step ([bos_id] + ids + [eos_id]):
Evaluates to [1] + [10, 25, 88] + [2]

[1, 10, 25, 88, 2]

In [ ]:
def pad_sequence(
    ids: List[int],
    max_len: int,
    pad_id: int = PAD_ID,
) -> List[int]:
    """
    Truncate or right-pad a sequence to exactly max_len tokens.

    Truncation removes tokens from the END (keeps BOS and early
    content, may drop EOS — acceptable for training purposes).

    Args:
        ids    : list of integer token IDs
        max_len: target length
        pad_id : value to pad with

    Returns:
        list of length max_len
    """

    if(len(ids) >= max_len): return ids[:max_len]

    return ids + [pad_id]*(max_len - len(ids))

Input length: 3

Tokens needed: $5 - 3 = 2$

Operation: Appends [0, 0] to the end.

Output: [1, 105, 2, 0, 0] (Length: 5)

In [ ]:
def build_tf_dataset(
    src_lines : List[str],
    tgt_lines : List[str],
    encode_fn : Callable[[str], List[int]],
    max_len   : int,
    batch_size: int,
    bos_id    : int = BOS_ID,
    eos_id    : int = EOS_ID,
    pad_id    : int = PAD_ID,
    shuffle   : bool = True,
    shuffle_seed: int = 42,
) -> tf.data.Dataset:
    """
    Build a batched, padded tf.data.Dataset for sequence-to-sequence training.

    Each element yielded by the dataset is a tuple:
        (encoder_input, decoder_input, decoder_target)

    where:
        encoder_input  : shape (batch, max_len)      ← encoded source + BOS/EOS
        decoder_input  : shape (batch, max_len)      ← [BOS] + target (no EOS)
        decoder_target : shape (batch, max_len)      ← target + [EOS] (no BOS)

    Teacher forcing: the decoder sees its gold-standard prefix
    at every step (decoder_input), and is trained to predict
    the next token (decoder_target).

    Example for "Ich bin ein Student .":
        Full encoded target with BOS/EOS:
            [BOS, "Ich", "bin", "ein", "Student", ".", EOS]

        decoder_input  (fed to decoder):
            [BOS, "Ich", "bin", "ein", "Student", "."]

        decoder_target (what decoder must predict):
            ["Ich", "bin", "ein", "Student", ".", EOS]

    Args:
        src_lines    : list of raw source strings
        tgt_lines    : list of raw target strings
        encode_fn    : callable text → list[int]
        max_len      : sequence length (for both src and tgt)
        batch_size   : number of examples per batch
        bos_id       : BOS token ID
        eos_id       : EOS token ID
        pad_id       : PAD token ID (also used as 0-padding)
        shuffle      : whether to shuffle training data
        shuffle_seed : random seed for shuffle

    Returns:
        tf.data.Dataset of (encoder_input, decoder_input, decoder_target)
    """
    encoder_inputs  = []
    decoder_inputs  = []
    decoder_targets = []

    for src, tgt in zip (src_lines, tgt_lines):

        # 1. Source Encoding: Wraps raw source sentence in <BOS> and <EOS>, then pads/truncates it to max_len.
        src_ids = encode_and_wrap(src, encode_fn, bos_id, eos_id)
        src_ids = pad_sequence(src_ids, max_len, pad_id)

        # 2. Target Slicing & Shift:

            # tgt_full[:-1] drops the last token (drops <EOS>), serving as decoder_input.

            # tgt_full[1:] drops the first token (drops <BOS>), serving as decoder_target.

        tgt_full = encode_and_wrap(tgt, encode_fn, bos_id, eos_id)

        tgt_in  = pad_sequence(tgt_full[:-1], max_len, pad_id)  # drop last
        tgt_out = pad_sequence(tgt_full[1:],  max_len, pad_id)  # drop first

        encoder_inputs.append(src_ids)
        decoder_inputs.append(tgt_in)
        decoder_targets.append(tgt_out)

    encoder_inputs  = np.array(encoder_inputs,  dtype=np.int32)
    decoder_inputs  = np.array(decoder_inputs,  dtype=np.int32)
    decoder_targets = np.array(decoder_targets, dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((
        encoder_inputs,
        decoder_inputs,
        decoder_targets,
    ))

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(src_lines),
            seed=shuffle_seed,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset





$$(\text{encoder\_input}_i, \text{decoder\_input}_i, \text{decoder\_target}_i)$$

$$\begin{aligned} \text{encoder_input}  &= [\text{BOS}, \text{"I"}, \text{"am"}, \text{"happy"}, \text{EOS}, \text{PAD}] \\ \text{decoder_input}  &= [\text{BOS}, \text{"Ich"}, \text{"bin"}, \text{"froh"}, \text{PAD}, \text{PAD}] \\ \text{decoder_target} &= [\text{"Ich"}, \text{"bin"}, \text{"froh"}, \text{EOS}, \text{PAD}, \text{PAD}] \end{aligned}$$

# 7 - Mask Generation

Two masks are required by the Transformer:

1. **Padding Mask**
   - Prevents attention to `<PAD>` tokens (`ID = 0`).
   - Shape: `(batch, 1, 1, seq_len)`
   - Used in:
     - Encoder self-attention
     - Decoder cross-attention

2. **Look-Ahead (Causal) Mask**
   - Prevents the decoder from seeing future tokens.
   - Shape: `(1, 1, seq_len, seq_len)`
   - Uses a lower-triangular structure.
   - Position `i` can only attend to positions `0..i`.

3. **Combined Decoder Mask**
   - Element-wise maximum of the look-ahead mask and the decoder padding mask.
   - Used in decoder masked self-attention.

**Mask Convention:**

- `1.0` → **MASKED** (ignored)
- `0.0` → **NOT MASKED** (attended to)

In scaled dot-product attention, masked positions receive a large negative value (`−1e9`) before the softmax operation. This drives their attention weight to approximately `0`.

In [ ]:
def create_padding_mask(seq: tf.Tensor) -> tf.Tensor:
    """
    Create a binary mask that marks padding positions.

    Args:
        seq: integer token ID tensor of shape (batch, seq_len)

    Returns:
        mask: float32 tensor of shape (batch, 1, 1, seq_len)
              1.0 at PAD positions, 0.0 elsewhere

    Example:
        seq = [[3, 5, 0, 0]]           (2 real tokens, 2 PADs)
        mask → [[[[0., 0., 1., 1.]]]]  (1 = masked)
    """

    mask = tf.cast(tf.equal(seq, PAD_ID), dtype = tf.float32)
    # shape: (batch, seq_len)
    mask = mask[:, tf.newaxis, tf.newaxis, :]
    # shape: (batch, 1, 1, seq_len)
    return mask

In [ ]:
def create_look_ahead_mask(size: int) -> tf.Tensor:
    """
    Create a lower-triangular causal mask for decoder self-attention.

    Position i may only attend to positions 0..i.

    Args:
        size: sequence length

    Returns:
        mask: float32 tensor of shape (1, 1, size, size)
              0.0 at positions that may be attended to,
              1.0 at positions that must be masked

    Example (size=4):
        [[0, 1, 1, 1],
         [0, 0, 1, 1],
         [0, 0, 0, 1],
         [0, 0, 0, 0]]
    """
    ones = tf.ones((size, size))

    # tf.linalg.band_part(ones, -1, 0) creates a lower triangular matrix (including diagonal) = 1.0
    lower_tri = tf.linalg.band_part(ones, -1, 0)

    # We want the UPPER triangle (excluding diagonal) to be masked (value 1.0).
    mask = 1.0 - lower_tri

    # Expand for batch and head broadcasting
    mask = mask[tf.newaxis, tf.newaxis, :, :]

    return mask

In [ ]:
def create_decoder_mask(
    tgt_seq   : tf.Tensor,
    tgt_len   : int,
) -> tf.Tensor:
    """
    Combine look-ahead mask with target sequence padding mask.

    The decoder self-attention mask must enforce two things:
      1. Cannot attend to future positions (causal).
      2. Cannot attend to PAD positions.

    We take the element-wise maximum: a position is masked if
    EITHER condition requires it.

    Args:
        tgt_seq: int tensor of shape (batch, tgt_len)
        tgt_len: integer length of target sequence

    Returns:
        combined: float32 tensor of shape (batch, 1, tgt_len, tgt_len)
    """
    look_ahead = create_look_ahead_mask(tgt_len)
    # shape: (1, 1, tgt_len, tgt_len)

    tgt_pad = create_padding_mask(tgt_seq)
    # shape: (batch, 1, 1, tgt_len)

    # Broadcast and take max:
    # look_ahead : (1, 1, tgt_len, tgt_len)  — causal mask
    # tgt_pad    : (batch, 1, 1, tgt_len)    — padding on keys
    combined = tf.maximum(look_ahead, tgt_pad)
    # shape: (batch, 1, tgt_len, tgt_len)

    return combined

In [ ]:
def test_masks():
    print("Testing mask shapes and values …")

    # Test 1: padding mask
    seq = tf.constant([[1, 2, 3, 0, 0], [4, 0, 0, 0, 0]], dtype=tf.int32)
    pm = create_padding_mask(seq)
    assert pm.shape == (2, 1, 1, 5), f"Padding mask shape wrong: {pm.shape}"
    expected_pm_row0 = [0., 0., 0., 1., 1.]
    assert list(pm[0, 0, 0, :].numpy()) == expected_pm_row0, \
        f"Padding mask values wrong: {pm[0,0,0,:].numpy()}"
    print(f" Padding mask shape: {pm.shape}")
    print(f"     Row 0: {pm[0,0,0,:].numpy()}  (last 2 are PAD)")
    print(f"     Row 1: {pm[1,0,0,:].numpy()}  (last 4 are PAD)")

    # Test 2: look-ahead mask
    size = 4
    lm = create_look_ahead_mask(size)
    assert lm.shape == (1, 1, 4, 4), f"Look-ahead mask shape wrong: {lm.shape}"
    print(f"\n   Look-ahead mask shape: {lm.shape}")
    print(f"     Matrix (1=masked, lower-right triangle):")
    for row in lm[0, 0].numpy():
        print(f"       {row}")

    # Test 3: combined decoder mask
    tgt_seq = tf.constant([[1, 2, 3, 0], [1, 0, 0, 0]], dtype=tf.int32)
    dm = create_decoder_mask(tgt_seq, tgt_len=4)
    assert dm.shape == (2, 1, 4, 4), f"Decoder mask shape wrong: {dm.shape}"
    print(f"\n   Combined decoder mask shape: {dm.shape}")
    print(f"     Batch 0 (mask matrix, rows=queries, cols=keys):")
    for row in dm[0, 0].numpy():
        print(f"       {row}")

    # Verify causal property: position 0 must not see positions 1,2,3
    assert dm[0, 0, 0, 1].numpy() == 1.0, "Position 0 should not see position 1"
    assert dm[0, 0, 1, 0].numpy() == 0.0, "Position 1 should see position 0"
    print("\n   Causal property verified.")

    # Verify PAD masking: PAD positions should always be masked
    # In batch 0, position 3 is PAD → all queries should see it as masked
    assert dm[0, 0, 2, 3].numpy() == 1.0, "PAD at position 3 should be masked"
    print("   PAD masking verified.")

    print("\n All mask tests passed.")

In [ ]:
test_masks()

Testing mask shapes and values …
 Padding mask shape: (2, 1, 1, 5)
     Row 0: [0. 0. 0. 1. 1.]  (last 2 are PAD)
     Row 1: [0. 1. 1. 1. 1.]  (last 4 are PAD)

   Look-ahead mask shape: (1, 1, 4, 4)
     Matrix (1=masked, lower-right triangle):
       [0. 1. 1. 1.]
       [0. 0. 1. 1.]
       [0. 0. 0. 1.]
       [0. 0. 0. 0.]

   Combined decoder mask shape: (2, 1, 4, 4)
     Batch 0 (mask matrix, rows=queries, cols=keys):
       [0. 1. 1. 1.]
       [0. 0. 1. 1.]
       [0. 0. 0. 1.]
       [0. 0. 0. 1.]

   Causal property verified.
   PAD masking verified.

 All mask tests passed.


# 8 - Dummy Tokenizer — Pipeline Smoke Test

Before Phase 3 builds the real **BPE tokenizer**, we verify the entire data pipeline end-to-end using a simple **whitespace tokenizer** with a manually constructed vocabulary.

This proves that the following components work correctly:

**File Reading → Encoding → BOS/EOS → Padding → Batching → Mask Generation → Shape Verification**

The real tokenizer will later slot into the pipeline without requiring changes to the rest of the data-loading infrastructure.

In [ ]:
from collections import Counter

In [ ]:
STAGE_A_DIR

'/content/drive/MyDrive/Transformer_Mine/datasets/stage_a'

In [ ]:
src_lines_a, tgt_lines_a = load_raw_pairs(
    os.path.join(STAGE_A_DIR, 'train.en'),
    os.path.join(STAGE_A_DIR, 'train.de'),
)

In [ ]:
all_tokens = []

In [ ]:
for line in src_lines_a + tgt_lines_a:
    all_tokens.extend(line.split())

In [ ]:
len(all_tokens)

183

In [ ]:
token_counts = Counter(all_tokens)

In [ ]:
token_counts

Counter({'I': 5,
         'am': 2,
         'a': 6,
         'student': 1,
         '.': 40,
         'She': 5,
         'is': 7,
         'teacher': 1,
         'He': 4,
         'doctor': 1,
         'We': 2,
         'are': 2,
         'friends': 1,
         'They': 1,
         'students': 1,
         'like': 1,
         'music': 1,
         'likes': 2,
         'books': 1,
         'sports': 1,
         'go': 1,
         'to': 2,
         'school': 1,
         'goes': 2,
         'work': 1,
         'home': 1,
         'have': 2,
         'dog': 2,
         'has': 1,
         'cat': 2,
         'car': 1,
         'happy': 1,
         'sad': 1,
         'tired': 1,
         'The': 3,
         'small': 1,
         'big': 1,
         'book': 1,
         'new': 1,
         'Ich': 5,
         'bin': 2,
         'ein': 3,
         'Student': 1,
         'Sie': 6,
         'ist': 7,
         'eine': 2,
         'Lehrerin': 1,
         'Er': 4,
         'Arzt': 1,
         'Wir': 2,
      

In [ ]:
SPECIAL_TOKENS = ['<pad>', '<bos>', '<eos>', '<unk>']

In [ ]:
vocab_tokens   = SPECIAL_TOKENS + [t for t, _ in token_counts.most_common()]

In [ ]:
token2id       = {t: i for i, t in enumerate(vocab_tokens)}
id2token       = {i: t for t, i in token2id.items()}

In [ ]:
VOCAB_SIZE_DUMMY = len(vocab_tokens)

In [ ]:
assert token2id['<pad>'] == PAD_ID == 0
assert token2id['<bos>'] == BOS_ID == 1
assert token2id['<eos>'] == EOS_ID == 2
assert token2id['<unk>'] == UNK_ID == 3

In [ ]:
print(f"Dummy vocabulary size: {VOCAB_SIZE_DUMMY}")
print(f"Special tokens: {SPECIAL_TOKENS}")
print(f"Sample tokens: {list(token2id.items())[4:12]}")

Dummy vocabulary size: 85
Special tokens: ['<pad>', '<bos>', '<eos>', '<unk>']
Sample tokens: [('.', 4), ('is', 5), ('ist', 6), ('a', 7), ('Sie', 8), ('I', 9), ('She', 10), ('Ich', 11)]


In [ ]:
def dummy_encode(text: str) -> list:
    """Whitespace-split text and map tokens to IDs."""
    return [token2id.get(tok, UNK_ID) for tok in text.split()]

def dummy_decode(ids: list) -> str:
    """Map IDs back to tokens, skipping special tokens."""
    return ' '.join(
        id2token.get(i, '<unk>')
        for i in ids
        if i not in (PAD_ID, BOS_ID, EOS_ID)
    )

In [ ]:
MAX_LEN_A   = 20
BATCH_SIZE_A = 4


In [ ]:
print(f"\nBuilding Stage A dataset (max_len={MAX_LEN_A}, batch={BATCH_SIZE_A}) …")

dataset_a = build_tf_dataset(
    src_lines    = src_lines_a,
    tgt_lines    = tgt_lines_a,
    encode_fn    = dummy_encode,
    max_len      = MAX_LEN_A,
    batch_size   = BATCH_SIZE_A,
    bos_id       = BOS_ID,
    eos_id       = EOS_ID,
    pad_id       = PAD_ID,
    shuffle      = True,
    shuffle_seed = 42,
)


Building Stage A dataset (max_len=20, batch=4) …


In [ ]:
for enc_in, dec_in, dec_tgt in dataset_a.take(1):
    print(f"\nFirst batch shapes:")
    print(f"  encoder_input  : {enc_in.shape}   # (batch, max_len)")
    print(f"  decoder_input  : {dec_in.shape}   # (batch, max_len)")
    print(f"  decoder_target : {dec_tgt.shape}  # (batch, max_len)")

    # Decode first example
    print(f"\nFirst example:")
    print(f"  encoder_input (raw IDs) : {enc_in[0].numpy().tolist()}")
    print(f"  encoder_input (decoded) : {dummy_decode(enc_in[0].numpy())}")
    print(f"  decoder_input (raw IDs) : {dec_in[0].numpy().tolist()}")
    print(f"  decoder_input (decoded) : {dummy_decode(dec_in[0].numpy())}")
    print(f"  decoder_target(raw IDs) : {dec_tgt[0].numpy().tolist()}")
    print(f"  decoder_target(decoded) : {dummy_decode(dec_tgt[0].numpy())}")

    # decoder_input[1:] should equal decoder_target[:-1]
    # (wherever neither is PAD and both are within sequence)
    # We check the first non-zero overlap window.
    dec_in_np  = dec_in[0].numpy()
    dec_tgt_np = dec_tgt[0].numpy()
    # dec_in  starts with BOS_ID
    assert dec_in_np[0] == BOS_ID, "decoder_input should start with BOS"
    # dec_tgt should NOT start with BOS_ID
    assert dec_tgt_np[0] != BOS_ID, "decoder_target should not start with BOS"
    print("\n   Teacher-forcing shift verified (dec_input starts with BOS)")

    enc_pad_mask = create_padding_mask(enc_in)
    dec_mask     = create_decoder_mask(dec_in, tgt_len=MAX_LEN_A)
    cross_mask   = create_padding_mask(enc_in)

    print(f"\nMask shapes:")
    print(f"  enc_pad_mask (encoder self-attn) : {enc_pad_mask.shape}")
    print(f"  dec_mask     (decoder self-attn) : {dec_mask.shape}")
    print(f"  cross_mask   (cross-attention)   : {cross_mask.shape}")

    # Verify dec_mask is causal at position 0
    assert dec_mask[0, 0, 0, 1].numpy() == 1.0, "Causal mask check failed"
    print("   Causal masking verified in batch")

print("\n End-to-end pipeline smoke test passed.")
print("   The real BPE tokenizer (Phase 3) will replace dummy_encode/dummy_decode.")


First batch shapes:
  encoder_input  : (4, 20)   # (batch, max_len)
  decoder_input  : (4, 20)   # (batch, max_len)
  decoder_target : (4, 20)  # (batch, max_len)

First example:
  encoder_input (raw IDs) : [1, 9, 40, 41, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  encoder_input (decoded) : I like music .
  decoder_input (raw IDs) : [1, 11, 16, 62, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  decoder_input (decoded) : Ich mag Musik .
  decoder_target(raw IDs) : [11, 16, 62, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  decoder_target(decoded) : Ich mag Musik .

   Teacher-forcing shift verified (dec_input starts with BOS)

Mask shapes:
  enc_pad_mask (encoder self-attn) : (4, 1, 1, 20)
  dec_mask     (decoder self-attn) : (4, 1, 20, 20)
  cross_mask   (cross-attention)   : (4, 1, 1, 20)
   Causal masking verified in batch

 End-to-end pipeline smoke test passed.
   The real BPE tokenizer (Phase 3) will replace dummy_encode/dummy_decode.


# 9 - Dataset Statistics and Memory Estimation

Before any training, estimate whether the active dataset configuration is likely to fit comfortably within Colab's available memory.

In [ ]:
import sys
import psutil

def dataset_statistics(src_path, tgt_path, max_len, batch_size, label=""):
    """
    Compute statistics for a parallel corpus and estimate memory usage.

    Reports:
      - sentence pair count
      - source / target token length distribution
      - percentage of sequences that exceed max_len (will be truncated)
      - estimated memory for the full numpy arrays (before tf.data)
      - estimated steps per epoch
    """
    print(f"\n{'='*60}")
    print(f"  Dataset statistics: {label}")
    print(f"{'='*60}")

    if not (os.path.exists(src_path) and os.path.exists(tgt_path)):
        print(f"  ⚠️  Files not yet downloaded for this stage.")
        print(f"      src: {src_path}")
        print(f"      tgt: {tgt_path}")
        return

    with open(src_path, encoding='utf-8') as f:
        src_lines = [l.strip() for l in f if l.strip()]
    with open(tgt_path, encoding='utf-8') as f:
        tgt_lines = [l.strip() for l in f if l.strip()]

    n_pairs = len(src_lines)
    print(f"  Sentence pairs    : {n_pairs:>10,}")

    src_lens = np.array([len(l.split()) for l in src_lines])
    tgt_lens = np.array([len(l.split()) for l in tgt_lines])

    for name, lens in [("Source (EN)", src_lens), ("Target (DE)", tgt_lens)]:
        pct_over = 100 * np.mean(lens > max_len)
        print(f"\n  {name}:")
        print(f"    Mean length       : {lens.mean():.1f}")
        print(f"    Median length     : {np.median(lens):.1f}")
        print(f"    95th percentile   : {np.percentile(lens, 95):.0f}")
        print(f"    Max length        : {lens.max()}")
        print(f"    Exceed max_len={max_len}: {pct_over:.1f}%  (will be truncated)")

    bytes_per_pair = 3 * max_len * 4  # 3 arrays × max_len × 4 bytes (int32)
    total_bytes    = n_pairs * bytes_per_pair
    total_mb       = total_bytes / 1_048_576

    print(f"\n  Memory estimate (numpy, pre-tf.data):")
    print(f"    Bytes per pair    : {bytes_per_pair}")
    print(f"    Total             : {total_mb:.1f} MB")

    ram = psutil.virtual_memory()
    ram_available_mb = ram.available / 1_048_576
    print(f"    Available RAM     : {ram_available_mb:.0f} MB")

    if total_mb > ram_available_mb * 0.5:
        print(f"  ⚠️  WARNING: Dataset may use more than 50% of available RAM.")
        print(f"              Consider using tf.data streaming from disk instead.")
    else:
        print(f"  ✅ Memory estimate is within safe limits.")

    # Steps per epoch
    steps = int(np.ceil(n_pairs / batch_size))
    print(f"\n  Training throughput:")
    print(f"    Batch size        : {batch_size}")
    print(f"    Steps per epoch   : {steps:,}")


for stage_key, cfg in DATASET_REGISTRY.items():
    dataset_statistics(
        src_path   = cfg['train_src'],
        tgt_path   = cfg['train_tgt'],
        max_len    = cfg['max_len'],
        batch_size = cfg['batch_size'],
        label      = cfg['name'],
    )

print(f"\n{'='*60}")
print("  System memory summary")
print(f"{'='*60}")
ram    = psutil.virtual_memory()
disk   = psutil.disk_usage('/content')
print(f"  RAM total     : {ram.total / 1e9:.1f} GB")
print(f"  RAM available : {ram.available / 1e9:.1f} GB")
print(f"  RAM used      : {ram.percent:.1f}%")
print(f"  Disk total    : {disk.total / 1e9:.1f} GB")
print(f"  Disk free     : {disk.free / 1e9:.1f} GB")


  Dataset statistics: Stage A — Tiny Handmade
  Sentence pairs    :         20

  Source (EN):
    Mean length       : 4.5
    Median length     : 5.0
    95th percentile   : 5
    Max length        : 5
    Exceed max_len=20: 0.0%  (will be truncated)

  Target (DE):
    Mean length       : 4.6
    Median length     : 5.0
    95th percentile   : 5
    Max length        : 5
    Exceed max_len=20: 0.0%  (will be truncated)

  Memory estimate (numpy, pre-tf.data):
    Bytes per pair    : 240
    Total             : 0.0 MB
    Available RAM     : 10700 MB
  ✅ Memory estimate is within safe limits.

  Training throughput:
    Batch size        : 4
    Steps per epoch   : 5

  Dataset statistics: Stage B — Tatoeba EN-DE
  Sentence pairs    :     44,581

  Source (EN):
    Mean length       : 3.4
    Median length     : 4.0
    95th percentile   : 4
    Max length        : 4
    Exceed max_len=100: 0.0%  (will be truncated)

  Target (DE):
    Mean length       : 3.5
    Median length     : 

# 10 - Save Pipeline Utilities to Drive

Save the data pipeline functions to a `.py` file on Drive so they can be imported in any future Colab session without re-running all the previous cells.

In [ ]:
# ============================================================
# PHASE 2 — CELL 10: Save Pipeline Utilities to Drive
# ============================================================
# Saves the data pipeline functions to a .py file on Drive
# so they can be imported in any future Colab session without
# re-running all the cells.

import os

UTILS_PATH = os.path.join(PROJECT_ROOT, 'data_pipeline.py')

code = '''"""
data_pipeline.py — Transformer Reproduction Project
Phase 2: Data loading and batching utilities.

Usage in a new Colab session:
    import sys
    sys.path.insert(0, "/content/drive/MyDrive/transformer_reproduction")
    from data_pipeline import (
        load_raw_pairs, encode_and_wrap, pad_sequence,
        build_tf_dataset, create_padding_mask,
        create_look_ahead_mask, create_decoder_mask,
    )
"""

import numpy as np
import tensorflow as tf
from typing import Callable, List, Tuple

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
UNK_ID = 3


def load_raw_pairs(src_path, tgt_path):
    with open(src_path, encoding="utf-8") as f:
        src = [l.strip() for l in f if l.strip()]
    with open(tgt_path, encoding="utf-8") as f:
        tgt = [l.strip() for l in f if l.strip()]
    assert len(src) == len(tgt)
    return src, tgt


def encode_and_wrap(text, encode_fn, bos_id=BOS_ID, eos_id=EOS_ID):
    return [bos_id] + encode_fn(text) + [eos_id]


def pad_sequence(ids, max_len, pad_id=PAD_ID):
    if len(ids) >= max_len:
        return ids[:max_len]
    return ids + [pad_id] * (max_len - len(ids))


def build_tf_dataset(src_lines, tgt_lines, encode_fn, max_len,
                     batch_size, bos_id=BOS_ID, eos_id=EOS_ID,
                     pad_id=PAD_ID, shuffle=True, shuffle_seed=42):
    enc_inputs, dec_inputs, dec_targets = [], [], []
    for src, tgt in zip(src_lines, tgt_lines):
        src_ids  = encode_and_wrap(src, encode_fn, bos_id, eos_id)
        src_ids  = pad_sequence(src_ids, max_len, pad_id)
        tgt_full = encode_and_wrap(tgt, encode_fn, bos_id, eos_id)
        tgt_in   = pad_sequence(tgt_full[:-1], max_len, pad_id)
        tgt_out  = pad_sequence(tgt_full[1:],  max_len, pad_id)
        enc_inputs.append(src_ids)
        dec_inputs.append(tgt_in)
        dec_targets.append(tgt_out)
    enc_inputs  = np.array(enc_inputs,  dtype=np.int32)
    dec_inputs  = np.array(dec_inputs,  dtype=np.int32)
    dec_targets = np.array(dec_targets, dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((enc_inputs, dec_inputs, dec_targets))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(src_lines), seed=shuffle_seed,
                        reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


def create_padding_mask(seq):
    mask = tf.cast(tf.equal(seq, 0), dtype=tf.float32)
    return mask[:, tf.newaxis, tf.newaxis, :]


def create_look_ahead_mask(size):
    ones      = tf.ones((size, size))
    upper_tri = tf.linalg.band_part(ones, 0, -1)
    mask      = 1.0 - upper_tri
    return mask[tf.newaxis, tf.newaxis, :, :]


def create_decoder_mask(tgt_seq, tgt_len):
    look_ahead = create_look_ahead_mask(tgt_len)
    tgt_pad    = create_padding_mask(tgt_seq)
    return tf.maximum(look_ahead, tgt_pad)
'''

with open(UTILS_PATH, 'w', encoding='utf-8') as f:
    f.write(code)

print(f"✅ Pipeline utilities saved to:")
print(f"   {UTILS_PATH}")
print()
print("To reload in a new session:")
print("    import sys")
print(f'    sys.path.insert(0, "{PROJECT_ROOT}")')
print("    from data_pipeline import *")

✅ Pipeline utilities saved to:
   /content/drive/MyDrive/Transformer_Mine/data_pipeline.py

To reload in a new session:
    import sys
    sys.path.insert(0, "/content/drive/MyDrive/Transformer_Mine")
    from data_pipeline import *
